In [ ]:
# ----------------------------
# Imports & Setup
# ----------------------------

import pandas as pd
from huggingface_hub import InferenceClient

HF_TOKEN = "myToken"  
client = InferenceClient(token=HF_TOKEN)

# ----------------------------
#Data
# ----------------------------

df = spark.read.format("delta").table("workspace.gold.abs_segments").toPandas()
cluster_profile = df.copy()

cluster_profile.index = [0, 1, 2, 3]  

# round for readability
cluster_profile = cluster_profile.round(2)


# ----------------------------
#Persona Generation Function (fixed)
# ----------------------------
def generate_persona(row, cluster_id):
    prompt = f"""You are a senior marketing strategist.

Generate a realistic, professional marketing persona for Cluster {cluster_id}.

Data:
- ---

Requirements:
- ---
"""
    try:
        response = client.chat_completion(
            model="model1"  ,
            messages=[
                {"role": "system", "content": "You are a senior marketing strategist."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=512,
            temperature=0.7
        )
        return response.choices[0].message.content
    except Exception as e:
        return f" Error: {str(e)}"

# ----------------------------
# Generate All Personas
# ----------------------------
dynamic_personas = {}

for cid in cluster_profile.index:
    print(f" Generating persona for Cluster {cid}...")
    row = cluster_profile.loc[cid]
    dynamic_personas[cid] = generate_persona(row, cid)
    print(f" Cluster {cid} done!\n")

# ----------------------------
# Save to Databricks Delta Table
# ----------------------------
results_df = pd.DataFrame({
    "cluster_id": list(dynamic_personas.keys()),
    "persona":    list(dynamic_personas.values())
})

spark_df = spark.createDataFrame(results_df)
spark_df.write.mode("overwrite").saveAsTable("abs_personas")

spark.table("workspace.default.abs_personas") \
     .write \
     .mode("overwrite") \
     .saveAsTable("workspace.gold.abs_personas")


,cluster_id,persona
0,0,Marketing Persona: \nName: Ethan Thompson\nJob...
1,1,**Marketing Persona: Cluster 1**\n\n**Name:** ...
2,2,**Marketing Persona: Cluster 2**\n\n**Name:** ...
3,3,**Marketing Persona: Cluster 3**\n\n**Name:** ...
